# Verifier QLoRA smoke test

Checks that the claim verifier's training pipeline runs on a Kaggle T4: Qwen2.5-3B-Instruct
loaded in 4-bit (float16 compute), LoRA adapters, a few training steps on 50 dummy
tone-paired examples, then generation on a few examples. The numbers say nothing about
verifier quality; real training data comes from the training-only assignments.

**Notebook settings:** Accelerator *GPU T4 x2*, Internet *on*.
Outputs go to `/kaggle/working/verifier_outputs/` (metrics, events, config; the LoRA adapter
is kept in the Kaggle output only, never committed).

In [ ]:
REPO_URL = "https://github.com/JagadishReddyK7/regrade-appeals-agent.git"
BRANCH = "main"
WORKDIR = "/kaggle/temp/regrade-appeals-agent"
OUT = "/kaggle/working/verifier_outputs"
CONFIG = "configs/verifier_smoke.yaml"

In [ ]:
import json
import os
import shutil
import subprocess

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, WORKDIR], check=True)
os.chdir(WORKDIR)
!git log -1 --oneline

In [ ]:
!pip install -q -e ".[gpu]"
!python -c "import sys, torch; print(sys.version, 'torch', torch.__version__, torch.version.cuda)"
!python -c "import torch; print('gpu', torch.cuda.is_available(), torch.cuda.get_device_name(0))"
!pip list 2>/dev/null | grep -iE "^(transformers|peft|bitsandbytes|accelerate) "

In [ ]:
def run(*args):
    done = subprocess.run(["python", "-m", *args], capture_output=True, text=True)
    print(done.stdout[-6000:], done.stderr[-6000:])
    return done.returncode


if run("verifier.dummy_data", "--out", "runs/verifier_dummy.jsonl", "--n", "50") != 0:
    raise RuntimeError("could not write the dummy data")

## Train (about 10 steps) and check generation

In [ ]:
code = run("verifier.train", "--config", CONFIG)
print("exit code", code)

## Save outputs

In [ ]:
os.makedirs(OUT, exist_ok=True)
for run_dir in sorted(os.listdir("runs")):
    path = os.path.join("runs", run_dir)
    if os.path.isdir(path):
        shutil.copytree(path, f"{OUT}/{run_dir}", dirs_exist_ok=True)
for path in sorted(os.listdir(OUT)):
    metrics = os.path.join(OUT, path, "metrics.json")
    if os.path.exists(metrics):
        print(json.dumps(json.load(open(metrics, encoding="utf-8")), indent=2))
!pip freeze > {OUT}/pip_freeze.txt
!git rev-parse HEAD > {OUT}/commit.txt
shutil.make_archive("/kaggle/working/verifier_outputs", "zip", OUT)